# Stored Momentum from the Environment

## Introduction

A spacecraft's reaction wheels absorb the angular impulse of every torque the
environment puts on it. Three torques matter in low Earth orbit: the Earth's
gravity gradient across a body whose moments of inertia differ, sunlight pressing
on its surfaces off the centre of mass, and aerodynamic drag doing the same at low
altitude. None of them is a timer. Each follows from where the spacecraft is, how
it is pointed and what it is made of, so the momentum its wheels store builds at a
rate the orbit and the attitude set, and the dumps that unload it should follow
from that build-up.

COASTSim tracks stored momentum from these torques as a run goes:

* `StoredMomentumTracker` evaluates each enabled torque source in the body frame at
  every step and integrates their sum in inertial coordinates, so a change of
  attitude alone never creates momentum. The stored vector is reported in the body
  frame with each source's torque.
* **Gravity gradient** needs the inertia tensor (`spacecraft_bus.inertia_tensor_body_kg_m2`).
  It is secular for an inertial hold: the body sees the radial direction sweep
  round once per orbit and the torque does not average to zero.
* **Solar radiation pressure** needs the spacecraft's surfaces: flat plates with an
  area, a normal, a position and reflectivities, about the centre of mass
  (`spacecraft_bus.surfaces`). With none configured, a placeholder model is built
  from a box, the solar panels with a geometry and the telescope apertures, of the
  right order and nothing more. It is secular whenever the Sun direction is fixed
  in the body, which an inertial or Sun-referenced hold makes it.
* **Aerodynamic drag** is not yet a source; the source list is where it goes.

This notebook enables both sources, runs a day of dispatch and a day of one held
attitude, and shows what fills the wheels: how much, from which term, and how the
answer depends on the attitude flown and on the centre of mass. Nothing is enforced
here: capacity limits, magnetorquer unloading and thruster dumps come next and
build on this tracking.

## Step 1: Import Libraries

In [ ]:
%matplotlib inline
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
from rust_ephem import EarthLimbConstraint, SunConstraint, TLEEphemeris

from conops import create_ditl
from conops.common.vector import attitude_to_quat, quaternion_to_rotation_matrix
from conops.config import (
    AttitudeControlSystem,
    Battery,
    Constraint,
    GroundStationRegistry,
    MissionConfig,
    PanelGeometry,
    Payload,
    RadiatorConfiguration,
    SolarPanel,
    SolarPanelSet,
    SpacecraftBus,
    StarTrackerConfiguration,
    StoredMomentumConfig,
    SurfaceModel,
    Telescope,
    TelescopeConfig,
    TelescopeType,
    box_surfaces,
    guess_surfaces,
    panel_surfaces,
    telescope_surfaces,
)
from conops.simulation.momentum import (
    SolarRadiationPressureTorque,
    StoredMomentumTracker,
)
from conops.targets import Pointing

## Step 2: Spacecraft Configuration and Ephemeris

A 600 km orbit from the example TLE, sampled every 10 seconds: the tracker refuses
coarser samples, and with a 0.5 deg/s slew rate its five-degree guard allows exactly
10 s. The spacecraft has an inertia tensor of 1000, 1200 and 800 kg m² about its
body axes, a 0.5 m imager with a 1.2 m tube along the boresight (body +X), and one
3 m by 1.2 m array on the +Y side whose `geometry` gives it an area and a position.
Both torque sources are enabled; no surfaces are configured, so the placeholder
model is built from the bus, the array and the aperture.

In [ ]:
begin = datetime(2025, 12, 1, tzinfo=timezone.utc)
end = begin + timedelta(hours=24)
ephemeris = TLEEphemeris(begin=begin, end=end, step_size=10, tle="example.tle")

INERTIA = [[1000.0, 0.0, 0.0], [0.0, 1200.0, 0.0], [0.0, 0.0, 800.0]]


def make_config(
    gravity_gradient: bool = True, solar_radiation_pressure: bool = True
) -> MissionConfig:
    config = MissionConfig(
        ground_stations=GroundStationRegistry(stations=[]),
        constraint=Constraint(
            sun_constraint=SunConstraint(min_angle=45),
            earth_constraint=EarthLimbConstraint(min_angle=10),
        ),
        solar_panel=SolarPanelSet(
            panels=[
                SolarPanel(
                    name="Array",
                    normal=(0.0, 1.0, 0.0),
                    max_power=600.0,
                    geometry=PanelGeometry(
                        center_m=(0.0, 1.8, 0.0),
                        u=(1.0, 0.0, 0.0),
                        v=(0.0, 0.0, 1.0),
                        width_m=3.0,
                        height_m=1.2,
                    ),
                )
            ]
        ),
        battery=Battery(watthour=100_000.0),
        spacecraft_bus=SpacecraftBus(
            inertia_tensor_body_kg_m2=INERTIA,
            attitude_control=AttitudeControlSystem(
                max_slew_rate=0.5,
                slew_acceleration=0.25,
                settle_time=30.0,
                stored_momentum=StoredMomentumConfig(
                    gravity_gradient_enabled=gravity_gradient,
                    solar_radiation_pressure_enabled=solar_radiation_pressure,
                ),
            ),
            star_trackers=StarTrackerConfiguration(
                star_trackers=[], min_functional_trackers=0, modes_require_lock=[]
            ),
            radiators=RadiatorConfiguration(radiators=[]),
        ),
        payload=Payload(
            instruments=[
                Telescope(
                    name="Imager",
                    optics=TelescopeConfig(
                        aperture_m=0.5,
                        focal_length_m=3.5,
                        telescope_type=TelescopeType.RITCHEY_CHRETIEN,
                        tube_length_m=1.2,
                    ),
                )
            ]
        ),
    )
    config.constraint.ephem = ephemeris
    return config


config = make_config()

## Step 3: The Surfaces Sunlight Acts On

`guess_surfaces` is what a run uses when `spacecraft_bus.surfaces` is None: the six
faces of a 1.5 m by 1.5 m by 2 m box at the body origin, both faces of the array
(the cell side mostly absorbing, the back more reflective), and the aperture as an
absorbing disc at the end of the tube, all about a centre of mass 5 cm behind the
boresight. A mission's mass properties replace it with `SurfaceModel` directly, or
with `box_surfaces`, `panel_surfaces` and `telescope_surfaces` and a measured centre
of mass.

In [ ]:
surfaces = guess_surfaces(config.solar_panel, config.payload)
print(
    f"Centre of mass {surfaces.center_of_mass_m} m, {surfaces.total_area_m2:.2f} m² in all"
)
print(
    f"{'plate':<18} {'centre (m)':>22} {'normal':>18} {'area':>6} {'spec':>5} {'diff':>5}"
)
for plate in surfaces.surfaces:
    centre = ", ".join(f"{c:5.2f}" for c in plate.center_m)
    normal = ", ".join(f"{n:4.1f}" for n in plate.normal)
    print(
        f"{plate.name:<18} {centre:>22} {normal:>18} {plate.area_m2:6.2f} "
        f"{plate.specular_reflectivity:5.2f} {plate.diffuse_reflectivity:5.2f}"
    )

## Step 4: A Day of Dispatch

Forty targets with 20-minute snapshots, dispatched on merit by `QueueDITL`. The
spacecraft slews between them many times an hour, so no attitude is held for long
and neither torque is secular for long. The telemetry carries, at every step, the
stored vector in the body frame, its norm, and each source's torque by name.

In [ ]:
SNAPSHOT = 1200  # s


def make_targets(config: MissionConfig, seed: int) -> list[Pointing]:
    rng = np.random.default_rng(seed)
    targets = []
    for i in range(40):
        target = Pointing(
            config=config,
            ra=rng.uniform(0, 360),
            dec=rng.uniform(-60, 60),
            obsid=100 + i,
            name=f"Target {i}",
            merit=rng.uniform(20, 90),
            ss_min=SNAPSHOT,
            ss_max=SNAPSHOT,
        )
        target.exptime = 3 * SNAPSHOT
        targets.append(target)
    return targets


def run(config: MissionConfig, targets: list[Pointing]):
    ditl = create_ditl(config, targets, begin, end)
    ditl.step_size = 10
    assert ditl.calc()
    return ditl


dispatch = run(config, make_targets(config, 3))
hours = (
    np.array([hk.timestamp.timestamp() for hk in dispatch.telemetry.housekeeping])
    - begin.timestamp()
) / 3600
norm = np.array(dispatch.telemetry.housekeeping.stored_momentum_norm_n_m_s, dtype=float)
print(
    f"{len(hours)} samples; stored momentum ends at {norm[-1]:.2f} N m s, peak {norm.max():.2f} N m s"
)
sample = dispatch.telemetry.housekeeping[len(hours) // 2]
print("One sample's torques, in N m:")
for name, torque in sample.disturbance_torques_body_n_m.items():
    print(f"  {name:<26} {np.linalg.norm(torque):.2e}")

## Step 5: What Each Source Contributes

Each source's torque is reported in the body frame. Rotating it into inertial
coordinates with the attitude of the same sample and integrating gives each
source's share of the stored vector, which is what the tracker does with their sum.
The plot shows the norm of the total and of each source's part, with the eclipses
shaded: sunlight contributes nothing through them.

In [ ]:
def momentum_by_source(ditl) -> dict[str, np.ndarray]:
    """Each source's accumulated momentum in inertial coordinates, by name."""
    records = ditl.telemetry.housekeeping
    times = np.array([hk.timestamp.timestamp() for hk in records])
    names = list(records[0].disturbance_torques_body_n_m)
    torques_eci = {name: np.zeros((len(records), 3)) for name in names}
    for k, hk in enumerate(records):
        rotation = quaternion_to_rotation_matrix(
            [hk.quat_w, hk.quat_x, hk.quat_y, hk.quat_z]
        )
        for name in names:
            torques_eci[name][k] = rotation.T @ np.asarray(
                hk.disturbance_torques_body_n_m[name]
            )
    momentum = {}
    for name in names:
        impulse = np.zeros_like(torques_eci[name])
        steps = np.diff(times)
        impulse[1:] = np.cumsum(
            0.5 * (torques_eci[name][1:] + torques_eci[name][:-1]) * steps[:, None],
            axis=0,
        )
        momentum[name] = impulse
    return momentum


def shade_eclipses(ax, ditl, hours):
    eclipse = np.array(ditl.telemetry.housekeeping.in_eclipse, dtype=bool)
    edges = np.flatnonzero(np.diff(eclipse.astype(int)))
    starts = [0] if eclipse[0] else []
    starts += [k + 1 for k in edges if eclipse[k + 1]]
    stops = [k + 1 for k in edges if not eclipse[k + 1]]
    if eclipse[-1]:
        stops.append(len(eclipse) - 1)
    for s, e in zip(starts, stops):
        ax.axvspan(hours[s], hours[e], color="0.9", zorder=0)


by_source = momentum_by_source(dispatch)
fig, ax = plt.subplots(figsize=(10, 3.5))
shade_eclipses(ax, dispatch, hours)
ax.plot(hours, norm, color="k", label="total")
for name, momentum in by_source.items():
    ax.plot(hours, np.linalg.norm(momentum, axis=1), label=name.replace("_", " "))
ax.set_xlim(0, 24)
ax.set_xlabel("Hours")
ax.set_ylabel("Stored momentum (N m s)")
ax.set_title("A day of dispatch: what each torque leaves in the wheels")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
for name, momentum in by_source.items():
    print(f"{name:<26} {np.linalg.norm(momentum[-1]):6.2f} N m s at the end of the day")

## Step 6: An Attitude Held All Day

The same spacecraft holding one target through the whole day. For an inertial hold
the gravity gradient is secular: the radial direction sweeps round the body once an
orbit and the torque's average is not zero. The Sun direction is fixed in the body
too, so sunlight's torque is as secular, held off only by the eclipses. Compare the
two days: the varied dispatch schedule averages much of both torques away, the hold
does not.

In [ ]:
held_target = Pointing(
    config=config,
    ra=80.0,
    dec=-20.0,
    obsid=900,
    name="Held",
    merit=100.0,
    ss_min=600,
    ss_max=24 * 3600,
)
held_target.exptime = 24 * 3600
held = run(make_config(), [held_target])
held_hours = (
    np.array([hk.timestamp.timestamp() for hk in held.telemetry.housekeeping])
    - begin.timestamp()
) / 3600
held_norm = np.array(
    held.telemetry.housekeeping.stored_momentum_norm_n_m_s, dtype=float
)
held_by_source = momentum_by_source(held)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, (label, h, n, parts, run_) in zip(
    axes,
    [
        ("Dispatch", hours, norm, by_source, dispatch),
        ("Held attitude", held_hours, held_norm, held_by_source, held),
    ],
):
    shade_eclipses(ax, run_, h)
    ax.plot(h, n, color="k", label="total")
    for name, momentum in parts.items():
        ax.plot(h, np.linalg.norm(momentum, axis=1), label=name.replace("_", " "))
    ax.set_xlim(0, 24)
    ax.set_xlabel("Hours")
    ax.set_title(label)
axes[0].set_ylabel("Stored momentum (N m s)")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()
print(
    f"Dispatch ends at {norm[-1]:.2f} N m s, the held attitude at {held_norm[-1]:.2f} N m s"
)

## Step 7: Sunlight Depends on the Centre of Mass

Sunlight's torque is the force on each plate times its lever arm from the centre
of mass, so where the centre of mass sits decides how much a given spacecraft
takes. The tracker can be run on its own, with sunlight as its only source, at the
held attitude over the day's ephemeris: here with the centre of mass at the body
origin, 5 cm behind the boresight as the placeholder has it, and 20 cm behind.
The slope of each line is the secular rate that attitude gives.

In [ ]:
positions = np.asarray(ephemeris.gcrs_pv.position)
sun = np.asarray(ephemeris.sun_pv.position)
times = np.array([t.timestamp() for t in ephemeris.timestamp])
sunlit = ~np.array([config.constraint.in_eclipse(0.0, 0.0, float(t)) for t in times])
quaternion = attitude_to_quat(80.0, -20.0, 0.0)

fig, ax = plt.subplots(figsize=(10, 3.5))
for offset in (0.0, -0.05, -0.20):
    model = SurfaceModel(
        center_of_mass_m=(offset, 0.0, 0.0),
        surfaces=[
            *box_surfaces((1.5, 1.5, 2.0)),
            *panel_surfaces(config.solar_panel),
            *telescope_surfaces(config.payload),
        ],
    )
    tracker = StoredMomentumTracker(
        INERTIA, sources=[SolarRadiationPressureTorque(model)]
    )
    norms = [
        tracker.update(
            utime=float(times[k]),
            position_eci_km=positions[k],
            attitude_quaternion_eci_to_body=quaternion,
            sun_eci_km=sun[k] - positions[k],
            sunlit=bool(sunlit[k]),
        ).stored_momentum_norm_n_m_s
        for k in range(len(times))
    ]
    ax.plot(
        (times - begin.timestamp()) / 3600,
        norms,
        label=f"centre of mass {offset * 100:+.0f} cm",
    )
ax.set_xlim(0, 24)
ax.set_xlabel("Hours")
ax.set_ylabel("Stored momentum from sunlight (N m s)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Summary

* Stored momentum is integrated from the torques the environment actually applies,
  gravity gradient on the inertia tensor and sunlight on the surfaces, at the orbit
  position and attitude of every step; the telemetry reports the stored vector and
  each source's torque.
* How fast the wheels fill is a property of the schedule: a varied schedule
  averages the torques, an inertial hold makes them secular. Gravity gradient
  dominates for a spacecraft with hundreds of kg m² of inertia difference;
  sunlight's share is set by the centre of mass.
* The placeholder surface model gives the right order of magnitude; a mission's
  mass properties go in `spacecraft_bus.surfaces`.
* Drag, magnetorquer unloading, capacity limits and thruster dumps triggered by the
  predicted build-up are the next steps, and all of them build on this tracking.